In [1]:
import numpy as np

class DBSCAN:

    def __init__(self, eps=0.5, min_samples=5):
        self.eps = eps
        self.min_samples = min_samples

    def fit_predict(self, X):

        n = len(X)

        # -2 means unvisited
        # -1 means noise
        labels = np.full(n, -2)

        cluster_id = 0

        # Find neighbours of a point
        def get_neighbors(point_idx):

            distances = np.linalg.norm(
                X - X[point_idx], axis=1
            )

            return np.where(distances <= self.eps)[0]

        for i in range(n):

            if labels[i] != -2:
                continue

            neighbors = get_neighbors(i)

            # Not enough neighbours
            if len(neighbors) < self.min_samples:
                labels[i] = -1
                continue

            # Start a new cluster
            labels[i] = cluster_id

            seeds = list(neighbors)
            j = 0

            while j < len(seeds):

                point = seeds[j]

                # Previously marked noise can become border
                if labels[point] == -1:
                    labels[point] = cluster_id

                if labels[point] != -2:
                    j += 1
                    continue

                labels[point] = cluster_id

                new_neighbors = get_neighbors(point)

                # Expand only from core points
                if len(new_neighbors) >= self.min_samples:

                    for neighbor in new_neighbors:
                        if labels[neighbor] in (-2, -1):
                            seeds.append(neighbor)

                j += 1

            cluster_id += 1

        self.labels_ = labels
        return labels

In [2]:
X = np.array([
    [1, 1],
    [1.1, 1],
    [1.2, 1.1],
    [5, 5],
    [5.1, 5],
    [9, 9]
])

model = DBSCAN(
    eps=0.3,
    min_samples=2
)

labels = model.fit_predict(X)

print(labels)

[ 0  0  0  1  1 -1]


In [3]:
from sklearn.cluster import DBSCAN
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

model = make_pipeline(
    StandardScaler(),
    DBSCAN(
        eps=0.5,
        min_samples=5,
        metric='euclidean'
    )
)

labels = model.fit_predict(X)